# Output Label: `support_status` dan `priority_segment`

Notebook ini dibuat khusus untuk menghasilkan label hasil analisis:

1. **`support_status`**: label utama untuk melihat apakah mahasiswa berada pada kondisi stabil, silent struggle, atau sudah mencari bantuan profesional.
2. **`priority_segment`**: label segmentasi prioritas untuk menentukan kelompok mahasiswa yang perlu diprioritaskan dalam rancangan Smart Campus.

Catatan penting: label ini **bukan diagnosis medis**, tetapi label segmentasi berbasis data survei.

## 1. Import Library dan Load Dataset

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

base_dir = Path("/mnt/data") if Path("/mnt/data").exists() else Path.cwd()

data_path = base_dir / "Student Mental health.csv"
if not data_path.exists():
    data_path = Path("./dataset/Student Mental health.csv")

output_dir = base_dir / "label_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(data_path)

print("Dataset berhasil dibaca")
print("Jumlah baris:", df_raw.shape[0])
print("Jumlah kolom:", df_raw.shape[1])
display(df_raw.head())

Dataset berhasil dibaca
Jumlah baris: 101
Jumlah kolom: 11


,Timestamp,Choose your gender,Age,What is your course?,Your current year of Study,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,8/7/2020 12:02,Female,18.0,Engineering,year 1,3.00 - 3.49,No,Yes,No,Yes,No
1,8/7/2020 12:04,Male,21.0,Islamic education,year 2,3.00 - 3.49,No,No,Yes,No,No
2,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,No,Yes,Yes,Yes,No
3,8/7/2020 12:06,Female,22.0,Laws,year 3,3.00 - 3.49,Yes,Yes,No,No,No
4,8/7/2020 12:13,Male,23.0,Mathemathics,year 4,3.00 - 3.49,No,No,No,No,No


## 2. Cleaning Ringkas

In [3]:
df = df_raw.copy()

df = df.rename(columns={
    "Timestamp": "timestamp",
    "Choose your gender": "gender",
    "Age": "age",
    "What is your course?": "course",
    "Your current year of Study": "year_study",
    "What is your CGPA?": "cgpa",
    "Marital status": "marital_status",
    "Do you have Depression?": "depression",
    "Do you have Anxiety?": "anxiety",
    "Do you have Panic attack?": "panic_attack",
    "Did you seek any specialist for a treatment?": "seek_specialist"
})

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())

df["year_study"] = (
    df["year_study"]
    .str.lower()
    .str.replace("year", "Year", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["cgpa"] = (
    df["cgpa"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["course_clean"] = (
    df["course"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.title()
)

course_map = {
    "Benl": "BENL",
    "Bcs": "BCS",
    "Bit": "BIT",
    "Irkhs": "IRKHS",
    "Koe": "KOE",
    "Econs": "Economics",
    "Engine": "Engineering",
    "Fiqh Fatwa": "Fiqh",
    "Kirkhs": "KIRKHS",
}
df["course_clean"] = df["course_clean"].replace(course_map)

yes_no_cols = ["depression", "anxiety", "panic_attack", "seek_specialist", "marital_status"]
for col in yes_no_cols:
    df[col] = df[col].astype(str).str.title().map({"Yes": 1, "No": 0})

df.insert(0, "student_id", [f"S{i:03d}" for i in range(1, len(df) + 1)])

display(df.head())

,student_id,timestamp,gender,age,course,year_study,cgpa,marital_status,depression,anxiety,panic_attack,seek_specialist,course_clean
0,S001,8/7/2020 12:02,Female,18.0,Engineering,Year 1,3.00 - 3.49,0,1,0,1,0,Engineering
1,S002,8/7/2020 12:04,Male,21.0,Islamic education,Year 2,3.00 - 3.49,0,0,1,0,0,Islamic Education
2,S003,8/7/2020 12:05,Male,19.0,BIT,Year 1,3.00 - 3.49,0,1,1,1,0,BIT
3,S004,8/7/2020 12:06,Female,22.0,Laws,Year 3,3.00 - 3.49,1,1,0,0,0,Laws
4,S005,8/7/2020 12:13,Male,23.0,Mathemathics,Year 4,3.00 - 3.49,0,0,0,0,0,Mathemathics


## 3. Membuat Label Analisis

In [4]:
issue_cols = ["depression", "anxiety", "panic_attack"]

df["issue_count"] = df[issue_cols].sum(axis=1)
df["has_any_issue"] = df["issue_count"] > 0
df["multiple_issue"] = df["issue_count"] >= 2

def make_support_status(row):
    if row["issue_count"] == 0:
        return "Stable / No Reported Indicator"
    elif row["seek_specialist"] == 1:
        return "Reached Support"
    else:
        return "Silent Struggle"

def make_priority_segment(row):
    if row["issue_count"] == 0:
        return "Stable"
    elif row["seek_specialist"] == 1:
        return "Supported"
    elif row["issue_count"] == 1:
        return "Low Concern"
    else:
        return "High Concern"

def make_indicator_combination(row):
    indicators = []
    if row["depression"] == 1:
        indicators.append("Depression")
    if row["anxiety"] == 1:
        indicators.append("Anxiety")
    if row["panic_attack"] == 1:
        indicators.append("Panic Attack")
    return " + ".join(indicators) if indicators else "No Indicator"

df["support_status"] = df.apply(make_support_status, axis=1)
df["priority_segment"] = df.apply(make_priority_segment, axis=1)
df["indicator_combination"] = df.apply(make_indicator_combination, axis=1)

display(df[[
    "student_id", "gender", "age", "course_clean", "year_study", "cgpa",
    "depression", "anxiety", "panic_attack", "seek_specialist",
    "issue_count", "indicator_combination", "support_status", "priority_segment"
]].head(15))

,student_id,gender,age,course_clean,year_study,cgpa,depression,anxiety,panic_attack,seek_specialist,issue_count,indicator_combination,support_status,priority_segment
0,S001,Female,18.0,Engineering,Year 1,3.00 - 3.49,1,0,1,0,2,Depression + Panic Attack,Silent Struggle,High Concern
1,S002,Male,21.0,Islamic Education,Year 2,3.00 - 3.49,0,1,0,0,1,Anxiety,Silent Struggle,Low Concern
2,S003,Male,19.0,BIT,Year 1,3.00 - 3.49,1,1,1,0,3,Depression + Anxiety + Panic Attack,Silent Struggle,High Concern
3,S004,Female,22.0,Laws,Year 3,3.00 - 3.49,1,0,0,0,1,Depression,Silent Struggle,Low Concern
4,S005,Male,23.0,Mathemathics,Year 4,3.00 - 3.49,0,0,0,0,0,No Indicator,Stable / No Reported Indicator,Stable
5,S006,Male,19.0,Engineering,Year 2,3.50 - 4.00,0,0,1,0,1,Panic Attack,Silent Struggle,Low Concern
6,S007,Female,23.0,Pendidikan Islam,Year 2,3.50 - 4.00,1,0,1,0,2,Depression + Panic Attack,Silent Struggle,High Concern
7,S008,Female,18.0,BCS,Year 1,3.50 - 4.00,0,1,0,0,1,Anxiety,Silent Struggle,Low Concern
8,S009,Female,19.0,Human Resources,Year 2,2.50 - 2.99,0,0,0,0,0,No Indicator,Stable / No Reported Indicator,Stable
9,S010,Male,18.0,IRKHS,Year 1,3.50 - 4.00,0,1,1,0,2,Anxiety + Panic Attack,Silent Struggle,High Concern


## 4. Ringkasan Hasil Label

In [5]:
support_summary = (
    df["support_status"]
    .value_counts()
    .rename_axis("support_status")
    .reset_index(name="count")
)
support_summary["percentage"] = (support_summary["count"] / len(df) * 100).round(2)

priority_order = ["Stable", "Low Concern", "High Concern", "Supported"]
priority_summary = (
    df["priority_segment"]
    .value_counts()
    .reindex(priority_order)
    .fillna(0)
    .astype(int)
    .rename_axis("priority_segment")
    .reset_index(name="count")
)
priority_summary["percentage"] = (priority_summary["count"] / len(df) * 100).round(2)

issue_count_summary = (
    df["issue_count"]
    .value_counts()
    .sort_index()
    .rename_axis("issue_count")
    .reset_index(name="count")
)
issue_count_summary["percentage"] = (issue_count_summary["count"] / len(df) * 100).round(2)

display(Markdown("### Support Status Summary"))
display(support_summary)

display(Markdown("### Priority Segment Summary"))
display(priority_summary)

display(Markdown("### Issue Count Summary"))
display(issue_count_summary)

### Support Status Summary

,support_status,count,percentage
0,Silent Struggle,58,57.43
1,Stable / No Reported Indicator,37,36.63
2,Reached Support,6,5.94


### Priority Segment Summary

,priority_segment,count,percentage
0,Stable,37,36.63
1,Low Concern,36,35.64
2,High Concern,22,21.78
3,Supported,6,5.94


### Issue Count Summary

,issue_count,count,percentage
0,0,37,36.63
1,1,36,35.64
2,2,18,17.82
3,3,10,9.90


## 5. Output Dataset dengan Label

In [6]:
final_columns = [
    "student_id", "timestamp", "gender", "age", "course", "course_clean",
    "year_study", "cgpa", "marital_status", "depression", "anxiety",
    "panic_attack", "seek_specialist", "issue_count", "has_any_issue",
    "multiple_issue", "indicator_combination", "support_status", "priority_segment"
]

df_labeled = df[final_columns].copy()

labeled_csv = output_dir / "student_mental_health_with_labels.csv"
support_csv = output_dir / "support_status_summary.csv"
priority_csv = output_dir / "priority_segment_summary.csv"
issue_csv = output_dir / "issue_count_summary.csv"

df_labeled.to_csv(labeled_csv, index=False)
support_summary.to_csv(support_csv, index=False)
priority_summary.to_csv(priority_csv, index=False)
issue_count_summary.to_csv(issue_csv, index=False)

print("Dataset berlabel berhasil disimpan:")
print(labeled_csv)

print("\nTabel summary berhasil disimpan:")
print(support_csv)
print(priority_csv)
print(issue_csv)

display(df_labeled.head(20))

Dataset berlabel berhasil disimpan:
c:\Users\MSI SWORD 16\Kuliah\Lomba\TEKRA\label_outputs\student_mental_health_with_labels.csv

Tabel summary berhasil disimpan:
c:\Users\MSI SWORD 16\Kuliah\Lomba\TEKRA\label_outputs\support_status_summary.csv
c:\Users\MSI SWORD 16\Kuliah\Lomba\TEKRA\label_outputs\priority_segment_summary.csv
c:\Users\MSI SWORD 16\Kuliah\Lomba\TEKRA\label_outputs\issue_count_summary.csv


,student_id,timestamp,gender,age,course,course_clean,year_study,cgpa,marital_status,depression,anxiety,panic_attack,seek_specialist,issue_count,has_any_issue,multiple_issue,indicator_combination,support_status,priority_segment
0,S001,8/7/2020 12:02,Female,18.0,Engineering,Engineering,Year 1,3.00 - 3.49,0,1,0,1,0,2,True,True,Depression + Panic Attack,Silent Struggle,High Concern
1,S002,8/7/2020 12:04,Male,21.0,Islamic education,Islamic Education,Year 2,3.00 - 3.49,0,0,1,0,0,1,True,False,Anxiety,Silent Struggle,Low Concern
2,S003,8/7/2020 12:05,Male,19.0,BIT,BIT,Year 1,3.00 - 3.49,0,1,1,1,0,3,True,True,Depression + Anxiety + Panic Attack,Silent Struggle,High Concern
3,S004,8/7/2020 12:06,Female,22.0,Laws,Laws,Year 3,3.00 - 3.49,1,1,0,0,0,1,True,False,Depression,Silent Struggle,Low Concern
4,S005,8/7/2020 12:13,Male,23.0,Mathemathics,Mathemathics,Year 4,3.00 - 3.49,0,0,0,0,0,0,False,False,No Indicator,Stable / No Reported Indicator,Stable
5,S006,8/7/2020 12:31,Male,19.0,Engineering,Engineering,Year 2,3.50 - 4.00,0,0,0,1,0,1,True,False,Panic Attack,Silent Struggle,Low Concern
6,S007,8/7/2020 12:32,Female,23.0,Pendidikan islam,Pendidikan Islam,Year 2,3.50 - 4.00,1,1,0,1,0,2,True,True,Depression + Panic Attack,Silent Struggle,High Concern
7,S008,8/7/2020 12:33,Female,18.0,BCS,BCS,Year 1,3.50 - 4.00,0,0,1,0,0,1,True,False,Anxiety,Silent Struggle,Low Concern
8,S009,8/7/2020 12:35,Female,19.0,Human Resources,Human Resources,Year 2,2.50 - 2.99,0,0,0,0,0,0,False,False,No Indicator,Stable / No Reported Indicator,Stable
9,S010,8/7/2020 12:39,Male,18.0,Irkhs,IRKHS,Year 1,3.50 - 4.00,0,0,1,1,0,2,True,True,Anxiety + Panic Attack,Silent Struggle,High Concern


## 6. Interpretasi Singkat Label

- **Stable / No Reported Indicator**: mahasiswa tidak melaporkan indikasi depression, anxiety, atau panic attack.
- **Silent Struggle**: mahasiswa memiliki minimal satu indikasi, tetapi belum mencari bantuan profesional.
- **Reached Support**: mahasiswa memiliki indikasi dan sudah mencari bantuan profesional.

Untuk prioritas:

- **Stable**: tidak ada indikasi.
- **Low Concern**: memiliki satu indikasi dan belum mencari bantuan.
- **High Concern**: memiliki dua atau tiga indikasi dan belum mencari bantuan.
- **Supported**: memiliki indikasi dan sudah mencari bantuan.

Label ini digunakan sebagai dasar analisis dan rekomendasi Smart Campus, bukan sebagai diagnosis medis.